# UCI Household Electric Power Consumption Benchmark

## 1. Overview

This notebook implements a complete, reproducible benchmark for short-term household electricity-load forecasting using the UCI Individual Household Electric Power Consumption dataset.

The established workflow is preserved: missing observations are removed, the target is `Global_active_power`, lag features 1–24 and calendar predictors are created, and an unshuffled chronological 80/20 split is used. The benchmark compares Persistence, Linear Regression, and Random Forest and saves all processed data, figures, models, predictions, metrics, environment details, and execution logs.

**Expected notebook location:** `UCI_Household/notebooks/UCI_Benchmark.ipynb`  
**Expected raw dataset:** `UCI_Household/data/raw/household_power_consumption.txt`


## 2. Import Libraries


In [ ]:
from datetime import datetime
from pathlib import Path
import platform
import sys
import time
import warnings

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("default")
BENCHMARK_START_TIME = time.perf_counter()
RUN_STARTED_AT = datetime.now().astimezone()

# Reproducible constants from the original forecasting workflow.
TARGET_COLUMN = "Global_active_power"
DATETIME_COLUMN = "Datetime"
TRAIN_FRACTION = 0.80
N_LAGS = 24
RANDOM_STATE = 42
RF_SAMPLE_SIZE = 300_000
RF_N_ESTIMATORS = 50
RF_MAX_DEPTH = 20
RF_MIN_SAMPLES_LEAF = 5
PLOT_SIZE = 1_440  # First 24 hours at one-minute frequency.
FIGURE_DPI = 300
MODEL_COMPRESSION_LEVEL = 3

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "font.size": 11,
})
print("Libraries imported and benchmark constants initialized.")


## 3. Define Project Paths


In [ ]:
# Jupyter normally uses the notebook's directory as Path.cwd().
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name.lower() != "notebooks":
    raise RuntimeError(
        "Run this notebook from the UCI_Household/notebooks directory. "
        f"Current working directory: {NOTEBOOK_DIR}"
    )

# Because the notebook is directly inside UCI_Household/notebooks, its parent is UCI_Household/.
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

RAW_DATA_PATH = RAW_DATA_DIR / "household_power_consumption.txt"
PROCESSED_DATA_PATH = PROCESSED_DIR / "UCI_household_processed.csv"
RUN_LOG_PATH = RESULTS_DIR / "run_log.txt"

for directory in (PROCESSED_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

# Start a fresh log for this run.
LOG_MESSAGES = []


def log_event(message):
    """Record, display, and immediately persist a timestamped event."""
    timestamp = datetime.now().astimezone().isoformat(timespec="seconds")
    entry = f"[{timestamp}] {message}"
    LOG_MESSAGES.append(entry)
    RUN_LOG_PATH.write_text("\n".join(LOG_MESSAGES) + "\n", encoding="utf-8")
    print(entry)


def save_figure(filename):
    """Save the active Matplotlib figure consistently and close it."""
    output_path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(output_path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()
    plt.close()
    log_event(f"Figure saved: {output_path.name}")


def save_model(model, destination):
    """Compress and atomically publish a model artifact."""
    temporary_path = destination.with_name(f"{destination.name}.tmp")
    if temporary_path.exists():
        temporary_path.unlink()
    try:
        joblib.dump(
            model,
            temporary_path,
            compress=("gzip", MODEL_COMPRESSION_LEVEL),
        )
        temporary_path.replace(destination)
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise


log_event(f"Benchmark started at {RUN_STARTED_AT.isoformat(timespec='seconds')}")
log_event(f"Project root: {PROJECT_ROOT}")
log_event(f"Python {platform.python_version()} on {platform.platform()}")


## 4. Load Dataset


In [ ]:
if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError(
        "UCI source dataset not found. Place household_power_consumption.txt at: "
        f"{RAW_DATA_PATH}"
    )

raw_df = pd.read_csv(
    RAW_DATA_PATH,
    sep=";",
    low_memory=False,
    na_values="?",
)
ORIGINAL_ROWS = len(raw_df)
log_event(f"Dataset loaded: {ORIGINAL_ROWS:,} rows and {raw_df.shape[1]} columns")
display(raw_df.head())


## 5. Data Inspection


In [ ]:
required_columns = {"Date", "Time", TARGET_COLUMN}
missing_columns = required_columns.difference(raw_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]}")
print("\nColumn names:")
print(raw_df.columns.tolist())
print("\nMissing values:")
display(raw_df.isna().sum().to_frame("Missing"))
print("\nDescriptive statistics:")
display(raw_df.describe(include="all"))
log_event("Initial data inspection completed")


## 6. Data Preprocessing

`Date` and `Time` are combined using the source dataset's day-first format. All measurement columns are converted to numeric values, and rows containing missing values are removed, matching the original notebook methodology.


In [ ]:
df = raw_df.copy()
df[DATETIME_COLUMN] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce",
)
df = df.drop(columns=["Date", "Time"])

measurement_columns = [column for column in df.columns if column != DATETIME_COLUMN]
df[measurement_columns] = df[measurement_columns].apply(pd.to_numeric, errors="coerce")

missing_before = int(df.isna().any(axis=1).sum())
df = df.dropna().sort_values(DATETIME_COLUMN, kind="stable").reset_index(drop=True)
PROCESSED_SOURCE_ROWS = len(df)

if df.empty:
    raise ValueError("No valid observations remain after preprocessing")
if (df[TARGET_COLUMN] <= 0).any():
    raise ValueError("Target contains non-positive values; MAPE would be undefined")

print(f"Rows removed because of missing/invalid values: {missing_before:,}")
print(f"Rows retained: {PROCESSED_SOURCE_ROWS:,}")
print(f"Time span: {df[DATETIME_COLUMN].min()} to {df[DATETIME_COLUMN].max()}")
display(df.head())
log_event(
    f"Preprocessing completed: {missing_before:,} incomplete rows removed; "
    f"{PROCESSED_SOURCE_ROWS:,} rows retained"
)


## 7. Exploratory Data Analysis


In [ ]:
# Global active power over time.
plt.figure(figsize=(16, 5))
plt.plot(df[DATETIME_COLUMN], df[TARGET_COLUMN], linewidth=0.3, color="#1f77b4")
plt.title("Global Active Power Over Time")
plt.xlabel("Date")
plt.ylabel("Global Active Power (kW)")
save_figure("time_series.png")

# Target distribution.
plt.figure(figsize=(9, 5))
plt.hist(df[TARGET_COLUMN], bins=50, color="#4c78a8", edgecolor="white")
plt.title("Distribution of Global Active Power")
plt.xlabel("Global Active Power (kW)")
plt.ylabel("Frequency")
save_figure("global_active_power_distribution.png")

# Measurement correlation matrix.
correlation = df[measurement_columns].corr()
plt.figure(figsize=(9, 7))
image = plt.imshow(correlation, interpolation="nearest", cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(image, label="Pearson correlation")
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=75, ha="right")
plt.yticks(range(len(correlation.columns)), correlation.columns)
plt.title("Correlation Matrix")
save_figure("correlation_heatmap.png")

display(correlation)
log_event("Exploratory data analysis completed")


## 8. Feature Engineering


In [ ]:
def create_features(frame, target_column, datetime_column, n_lags):
    """Create the original lag and calendar feature set."""
    featured = frame[[datetime_column, target_column]].copy()
    for lag in range(1, n_lags + 1):
        featured[f"lag_{lag}"] = featured[target_column].shift(lag)

    timestamps = featured[datetime_column].dt
    featured["hour"] = timestamps.hour
    featured["day"] = timestamps.day
    featured["dayofweek"] = timestamps.dayofweek
    featured["month"] = timestamps.month
    return featured.dropna().reset_index(drop=True)


model_df = create_features(df, TARGET_COLUMN, DATETIME_COLUMN, N_LAGS)
FEATURE_COLUMNS = [f"lag_{lag}" for lag in range(1, N_LAGS + 1)] + [
    "hour", "day", "dayofweek", "month"
]

if model_df.empty:
    raise ValueError("Feature engineering produced an empty dataset")
if model_df[FEATURE_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError("Missing values remain in the modeling dataset")

model_df.to_csv(PROCESSED_DATA_PATH, index=False)
if PROCESSED_DATA_PATH.stat().st_size == 0:
    raise IOError(f"Processed dataset is empty: {PROCESSED_DATA_PATH}")

print(f"Modeling rows: {len(model_df):,}")
print(f"Number of features: {len(FEATURE_COLUMNS)}")
display(model_df.head())
log_event(
    f"Feature engineering completed: {len(model_df):,} rows and "
    f"{len(FEATURE_COLUMNS)} features; processed dataset saved"
)


## 9. Train/Test Split


In [ ]:
split_index = int(len(model_df) * TRAIN_FRACTION)
if split_index <= 0 or split_index >= len(model_df):
    raise ValueError("Chronological split requires non-empty training and testing sets")

train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print(f"Training samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")
print(f"Training period: {train_df[DATETIME_COLUMN].iloc[0]} to {train_df[DATETIME_COLUMN].iloc[-1]}")
print(f"Testing period: {test_df[DATETIME_COLUMN].iloc[0]} to {test_df[DATETIME_COLUMN].iloc[-1]}")

plt.figure(figsize=(16, 5))
plt.plot(train_df[DATETIME_COLUMN], y_train, label="Training data", linewidth=0.3)
plt.plot(test_df[DATETIME_COLUMN], y_test, label="Testing data", linewidth=0.3)
plt.axvline(test_df[DATETIME_COLUMN].iloc[0], color="black", linestyle="--", label="80/20 split")
plt.title("Chronological Train/Test Split")
plt.xlabel("Date")
plt.ylabel("Global Active Power (kW)")
plt.legend()
save_figure("train_test_split.png")
log_event("Chronological 80/20 split completed without shuffling")


## 10. Baseline Model


In [ ]:
def evaluate_predictions(model_name, actual, predicted, training_time_seconds):
    """Calculate the shared benchmark metrics."""
    actual_array = np.asarray(actual, dtype=float)
    predicted_array = np.asarray(predicted, dtype=float)
    if actual_array.shape != predicted_array.shape:
        raise ValueError(f"Prediction shape mismatch for {model_name}")
    return {
        "Model": model_name,
        "MAE": mean_absolute_error(actual_array, predicted_array),
        "RMSE": mean_squared_error(actual_array, predicted_array) ** 0.5,
        "MAPE": mean_absolute_percentage_error(actual_array, predicted_array) * 100,
        "R2": r2_score(actual_array, predicted_array),
        "Training_Time_Seconds": float(training_time_seconds),
    }


def plot_predictions(predicted, title, filename):
    """Plot the first 24 hours of aligned test observations."""
    plot_count = min(PLOT_SIZE, len(y_test))
    plt.figure(figsize=(16, 5))
    plt.plot(
        test_df[DATETIME_COLUMN].iloc[:plot_count],
        y_test.iloc[:plot_count],
        label="Actual",
        linewidth=1,
    )
    plt.plot(
        test_df[DATETIME_COLUMN].iloc[:plot_count],
        np.asarray(predicted)[:plot_count],
        label="Forecast",
        linewidth=1,
    )
    plt.title(title)
    plt.xlabel("Datetime")
    plt.ylabel("Global Active Power (kW)")
    plt.legend()
    save_figure(filename)


metrics_records = []
predictions = {}

baseline_start = time.perf_counter()
y_pred_baseline = X_test["lag_1"].to_numpy()
baseline_training_time = time.perf_counter() - baseline_start
baseline_metrics = evaluate_predictions(
    "Persistence", y_test, y_pred_baseline, baseline_training_time
)
metrics_records.append(baseline_metrics)
predictions["Persistence_Prediction"] = y_pred_baseline

display(pd.DataFrame([baseline_metrics]).round(6))
plot_predictions(
    y_pred_baseline,
    "Persistence Forecast — First 24 Hours of Test Data",
    "persistence_prediction.png",
)
log_event(f"Persistence evaluated: RMSE={baseline_metrics['RMSE']:.6f}")


## 11. Linear Regression Model


In [ ]:
linear_regression = LinearRegression()
training_start = time.perf_counter()
linear_regression.fit(X_train, y_train)
linear_training_time = time.perf_counter() - training_start
y_pred_lr = linear_regression.predict(X_test)

linear_metrics = evaluate_predictions(
    "Linear Regression", y_test, y_pred_lr, linear_training_time
)
metrics_records.append(linear_metrics)
predictions["Linear_Regression_Prediction"] = y_pred_lr

display(pd.DataFrame([linear_metrics]).round(6))
plot_predictions(
    y_pred_lr,
    "Linear Regression Forecast — First 24 Hours of Test Data",
    "linear_regression_prediction.png",
)
log_event(
    f"Linear Regression trained in {linear_training_time:.3f} seconds; "
    f"RMSE={linear_metrics['RMSE']:.6f}"
)


## 12. Random Forest Model


In [ ]:
# Preserve the original representative-sample Random Forest methodology.
rf_sample_size = min(RF_SAMPLE_SIZE, len(X_train))
X_train_rf = X_train.sample(n=rf_sample_size, random_state=RANDOM_STATE)
y_train_rf = y_train.loc[X_train_rf.index]

random_forest = RandomForestRegressor(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print(f"Training Random Forest on {rf_sample_size:,} sampled training rows...")
training_start = time.perf_counter()
random_forest.fit(X_train_rf, y_train_rf)
rf_training_time = time.perf_counter() - training_start
y_pred_rf = random_forest.predict(X_test)

rf_metrics = evaluate_predictions(
    "Random Forest", y_test, y_pred_rf, rf_training_time
)
metrics_records.append(rf_metrics)
predictions["Random_Forest_Prediction"] = y_pred_rf

display(pd.DataFrame([rf_metrics]).round(6))
plot_predictions(
    y_pred_rf,
    "Random Forest Forecast — First 24 Hours of Test Data",
    "random_forest_prediction.png",
)
log_event(
    f"Random Forest trained on {rf_sample_size:,} rows in "
    f"{rf_training_time:.3f} seconds; RMSE={rf_metrics['RMSE']:.6f}"
)


## 13. Model Comparison


In [ ]:
results_df = pd.DataFrame(metrics_records).sort_values("RMSE", ignore_index=True)
display(results_df.style.format({
    "MAE": "{:.6f}",
    "RMSE": "{:.6f}",
    "MAPE": "{:.4f}%",
    "R2": "{:.6f}",
    "Training_Time_Seconds": "{:.3f}",
}))

plt.figure(figsize=(9, 5))
bars = plt.bar(results_df["Model"], results_df["RMSE"], color=["#4c78a8", "#f58518", "#54a24b"])
plt.bar_label(bars, fmt="%.4f", padding=3)
plt.ylabel("RMSE (kW; lower is better)")
plt.xlabel("Model")
plt.title("UCI Benchmark Model Comparison")
save_figure("model_comparison_rmse.png")
log_event(f"Model comparison completed; best model: {results_df.iloc[0]['Model']}")


## 14. Save Models and Results


In [ ]:
BENCHMARK_RESULTS_PATH = RESULTS_DIR / "benchmark_results.csv"
PREDICTIONS_PATH = RESULTS_DIR / "predictions.csv"
METRICS_PATH = RESULTS_DIR / "metrics.txt"
ENVIRONMENT_PATH = RESULTS_DIR / "environment.txt"
LINEAR_MODEL_PATH = MODELS_DIR / "linear_regression_model.pkl"
RF_MODEL_PATH = MODELS_DIR / "random_forest_model.pkl"

# Model artifacts are compressed to limit disk use on VM environments.
save_model(linear_regression, LINEAR_MODEL_PATH)
save_model(random_forest, RF_MODEL_PATH)

results_df.to_csv(BENCHMARK_RESULTS_PATH, index=False)
predictions_df = pd.DataFrame({
    DATETIME_COLUMN: test_df[DATETIME_COLUMN].to_numpy(),
    "Actual": y_test.to_numpy(),
    **predictions,
})
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

metrics_lines = ["UCI Household Benchmark Metrics", "=" * 80]
for record in metrics_records:
    metrics_lines.extend([
        "",
        f"Model: {record['Model']}",
        f"MAE: {record['MAE']:.6f}",
        f"RMSE: {record['RMSE']:.6f}",
        f"MAPE: {record['MAPE']:.6f}%",
        f"R²: {record['R2']:.6f}",
        f"Training time: {record['Training_Time_Seconds']:.6f} seconds",
    ])
METRICS_PATH.write_text("\n".join(metrics_lines) + "\n", encoding="utf-8")

TOTAL_RUNTIME = time.perf_counter() - BENCHMARK_START_TIME
environment_lines = [
    f"Python version: {platform.python_version()}",
    f"Platform: {platform.platform()}",
    f"pandas version: {pd.__version__}",
    f"numpy version: {np.__version__}",
    f"matplotlib version: {matplotlib.__version__}",
    f"scikit-learn version: {sklearn.__version__}",
    f"joblib version: {joblib.__version__}",
    "xgboost version: not used",
    f"Working directory: {NOTEBOOK_DIR}",
    f"Benchmark runtime: {TOTAL_RUNTIME:.6f} seconds",
]
ENVIRONMENT_PATH.write_text("\n".join(environment_lines) + "\n", encoding="utf-8")

for record in metrics_records:
    log_event(
        f"Final metrics — {record['Model']}: MAE={record['MAE']:.6f}, "
        f"RMSE={record['RMSE']:.6f}, MAPE={record['MAPE']:.6f}%, "
        f"R2={record['R2']:.6f}, training={record['Training_Time_Seconds']:.3f}s"
    )
log_event("Models trained: Persistence, Linear Regression, Random Forest")
log_event(f"Total benchmark runtime: {TOTAL_RUNTIME:.3f} seconds")

# Validate all required outputs before declaring success.
required_files = [
    PROCESSED_DATA_PATH,
    LINEAR_MODEL_PATH,
    RF_MODEL_PATH,
    BENCHMARK_RESULTS_PATH,
    PREDICTIONS_PATH,
    METRICS_PATH,
    ENVIRONMENT_PATH,
    RUN_LOG_PATH,
]
required_figures = [
    FIGURES_DIR / "time_series.png",
    FIGURES_DIR / "global_active_power_distribution.png",
    FIGURES_DIR / "correlation_heatmap.png",
    FIGURES_DIR / "train_test_split.png",
    FIGURES_DIR / "persistence_prediction.png",
    FIGURES_DIR / "linear_regression_prediction.png",
    FIGURES_DIR / "random_forest_prediction.png",
    FIGURES_DIR / "model_comparison_rmse.png",
]
validation_errors = [
    f"Missing or empty artifact: {path}"
    for path in required_files + required_figures
    if not path.is_file() or path.stat().st_size == 0
]
if len(pd.read_csv(BENCHMARK_RESULTS_PATH)) != 3:
    validation_errors.append("benchmark_results.csv must contain exactly three model rows")
expected_prediction_columns = [
    DATETIME_COLUMN,
    "Actual",
    "Persistence_Prediction",
    "Linear_Regression_Prediction",
    "Random_Forest_Prediction",
]
if pd.read_csv(PREDICTIONS_PATH, nrows=0).columns.tolist() != expected_prediction_columns:
    validation_errors.append("predictions.csv has incorrect columns")
if pd.read_csv(PROCESSED_DATA_PATH, nrows=1).empty:
    validation_errors.append("Processed feature dataset is empty")
if validation_errors:
    raise RuntimeError("Final validation failed:\n- " + "\n- ".join(validation_errors))

log_event("Final artifact validation passed")
print("FINAL VALIDATION PASSED")
print(f"Artifacts saved under: {PROJECT_ROOT}")


## 15. Conclusions


In [ ]:
best_model = results_df.iloc[0]
print("UCI HOUSEHOLD BENCHMARK SUMMARY")
print("=" * 80)
print(f"Original observations: {ORIGINAL_ROWS:,}")
print(f"Valid preprocessed observations: {PROCESSED_SOURCE_ROWS:,}")
print(f"Feature-dataset observations: {len(model_df):,}")
print(f"Number of predictors: {len(FEATURE_COLUMNS)}")
print(f"Training samples: {len(X_train):,}")
print(f"Testing samples: {len(X_test):,}")
print(f"Best-performing model by RMSE: {best_model['Model']}")
print(f"Best MAE: {best_model['MAE']:.6f} kW")
print(f"Best RMSE: {best_model['RMSE']:.6f} kW")
print(f"Best MAPE: {best_model['MAPE']:.6f}%")
print(f"Best R²: {best_model['R2']:.6f}")
print(f"Total runtime: {TOTAL_RUNTIME:.3f} seconds")
display(results_df)


The persistence forecast establishes the essential one-minute-ahead baseline. Linear Regression tests whether the same lag and calendar predictors provide a useful linear improvement, while Random Forest captures nonlinear relationships using the original representative training-sample strategy. The table above is generated from the untouched chronological test period and determines the best model by the lowest test RMSE.

All conclusions should be based on the values produced by the current full run rather than on hardcoded expectations about which model will win.
